In [ ]:
import os
import gc
import torch
import pickle
import codecs
import gensim
import numpy as np
import pandas as pd
import pickle as pkl
import torch.nn as nn
from tqdm import tqdm
import seaborn as sns
import torch.nn as nn
import lightgbm as lgb
from scipy import sparse
from typing import Tuple
import torch.nn.functional as F
from sklearn.metrics import log_loss
from text_unidecode import unidecode
from typing import Dict, List, Tuple
from transformers import AutoTokenizer
from sklearn.preprocessing import OneHotEncoder
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from transformers import AutoModel, AutoTokenizer, AutoConfig
import warnings; warnings.simplefilter('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


### Huge Ensemble
#### DeBERTa-Base + DeBERTa-Large + RoBERTa-Large + LightGBM(GoogleNews-W2V)
In this notebook, we use 4 different models to generate predictions.

**DeBERTa-Base:**
- Training [notebook](https://www.kaggle.com/code/yujikomi/train-custommodel) by [YUJI.K](https://www.kaggle.com/yujikomi)
- Concatenate Discourse Text, [SEP] and Essay Text
- Pass through Deberta-base-v3 model
- Apply Mean Pooling on the final hidden states
- Classify using CrossEntropyLoss into 3 classes
- Includes Dynamic Padding

**DeBERTa-Large:**
- Training [notebook](https://www.kaggle.com/code/brandonhu0215/feedback-deberta-large-lb0-619) by [DSML](https://www.kaggle.com/brandonhu0215)
- 512 max length
- WeightedLayerPooling (Slightly improve deberta-base model from simple [CLS] head)
- GroupFold
- Different learning rates across layers
- Preprocessing (encoding-resolve+normalize)

**RoBERTa-Large:**
- Training [notebook](https://www.kaggle.com/code/thedevastator/feedback-roberta-large-training) by [The Devastator](https://www.kaggle.com/thedevastator)
- 512 max len

**LightGBM(GoogleNews-W2V):**
- Training [notebook](https://www.kaggle.com/code/mujrush/feedback2-word2vec-lightgbm/notebook) by [MUJ!RUSH!](https://www.kaggle.com/mujrush)
- LightGBM model using Word2vec.
- Word2vec represents words in 300 dimensions. By averaging the 300-dimensional vectors of the words in the sentence, the sentence was represented in 300 dimensions.


<center>
    <img src="https://i.ibb.co/ygz26m4/power-rangers.png" style="max-height: 600px; border-radius:20px; border: 1px solid;">    
<center>

# Ensemble Configuration

In [ ]:
WEIGHTS = [0.20, 0.65, 0.05, 0.1]
MODEL_NAMES = ['deberta', 'deberta_large', 'roberta', 'lgbm']

# DeBERTa Models
_____

### Wait! what is so special about DeBERTa? 🤔

### DeBERTa V1

> [Paper](https://arxiv.org/pdf/2006.03654.pdf)
> [Official code](https://github.com/microsoft/DeBERTa)

DeBERTa: Decoding-enhanced BERT with disentangled attention: 

DeBERTa is a transformer-based neural language model that is (kind of) an improvement to RoBERTa.
It improves on BERT and RoBERTa models using two novel techniques. 


##### Disentangled self-attention mechanism

**The hidden issue of the transformer architecture**

Transformers process sequences assets, this makes them **permutation invariant**.
In simple terms: You can shuffle the tokens in the input and the transformer architecture will treat them the same way.

> **Proof that language is not permutation invariant:** 
> "I am a smart person" != "am I a smart person" (?)

![](https://i.ibb.co/YpFtJwq/0-7fsg-Ie1-Hgw-Pwgf-SD.png)

To solve this, we use positional encodings: We just add a sin/cos to the input so the transformer will know where exactly in the sequence the token is from.
But this creates another problem: Now the transformer is sensitive to the length of the input (and a bit to translation but leave this aside). 

A solution to this issue was proposed in the DeBERTa paper.

DeBERTa addresses this by using two learned vectors, which encode content and position, respectively.

##### The Enhanced Mask Decoder

The second novel technique is the Enhanced Mask Decoder, it incorporates absolute positions [of the sentence] in the decoding layer to predict the masked tokens in model pretraining.

![](https://i.ibb.co/TYTnFcP/0-x-V2-GV7-YX5u4ic47.png)

### DeBERTaV3

> [paper](https://arxiv.org/abs/2111.09543?context=cs)
> [Official code](https://github.com/microsoft/DeBERTa)


**In short:**

- Combine DeBERTa with ELECTRA-style training. 
- Employ a gradient-disentangled embedding sharing as one of the model's building blocks to avoid “tug-of-war” issues.


##### Electra's training

The authors replace the masked language modeling (MLM) with a more sample-efficient pretraining task: **replaced token detection (RTD)**, where the model is trained as a discriminator to **predict whether a token in the input had been corrupted**.
For replacing the token in the input, Electra trains a generator that creates adversarial noise that is supposed to "directly" be the input on which the discriminator needs to train on.

##### Gradient-disentangled embedding sharing (GDES)

In ELECTRA, the discriminator and the generator share the same token embeddings. This mechanism can however hurt training efficiency, as the training losses of the discriminator and the generator tend to pull token embeddings in different directions. 
The simple solution: The generator shares its embeddings with the discriminator but stops the gradients in the discriminator from backpropagating to the generator embeddings.


## Model 1: Deberta-Base

#### Configurations

In [ ]:
INPUT_DIR = '../input/feedback-prize-effectiveness/'

class CFG:
    CVs = []
    seed = 42
    lr = 3e-5
    epochs = 3
    n_fold = 5
    apex = True
    fast = True
    AMP = False
    n_splits = 5
    train = True
    wandb = False
    max_len = 512
    dropout = 0.1
    min_lr = 1e-6
    batch_size = 8
    freezing = True
    print_freq = 50
    target_size = 3
    num_workers = 0
    num_cycles = 0.5
    n_accumulate = 1
    scheduler = 'cosine'
    weigth_decay = 0.01
    num_warmup_steps = 0
    trn_fold = [0, 1, 2, 3, 4]
    gradient_checkpointing = True
    model = '../input/deberta-v3-base/deberta-v3-base'

#### Helper Function

In [ ]:
def criterion(outputs, labels):
    return nn.CrossEntropyLoss()(outputs, labels)

def softmax(z):
    assert len(z.shape) == 2
    s = np.max(z, axis=1)
    s = s[:, np.newaxis]
    e_x = np.exp(z - s)
    div = np.sum(e_x, axis=1)
    div = div[:, np.newaxis]
    return e_x / div

def freeze(module):
    for parameter in module.parameters():
        parameter.requires_grad = False
        
def get_freezed_parameters(module):
    freezed_parameters = []
    for name, parameter in module.named_parameters():
        if not parameter.requires_grad:
            freezed_parameters.append(name)
    return freezed_parameters

def get_essay(essay_id, is_train=True):
    parent_path = INPUT_DIR + 'train' if is_train else INPUT_DIR + 'test'
    essay_path = os.path.join(parent_path, f"{essay_id}.txt")
    essay_text = open(essay_path, 'r').read()
    return essay_text

#### Preprocessing

In [ ]:
# Testing Data
test = pd.read_csv(INPUT_DIR + 'test.csv')
test['essay_text'] = test['essay_id'].apply(lambda x: get_essay(x, is_train=False))

In [ ]:
if CFG.fast: tokenizer = AutoTokenizer.from_pretrained(CFG.model, use_fast=True)
else: tokenizer = AutoTokenizer.from_pretrained(CFG.model)
CFG.tokenizer = tokenizer

#### Normalization

In [ ]:
def replace_encoding_with_utf8(error: UnicodeError) -> Tuple[bytes, int]: return error.object[error.start : error.end].encode("utf-8"), error.end
def replace_decoding_with_cp1252(error: UnicodeError) -> Tuple[str, int]: return error.object[error.start : error.end].decode("cp1252"), error.end
codecs.register_error("replace_encoding_with_utf8", replace_encoding_with_utf8)
codecs.register_error("replace_decoding_with_cp1252", replace_decoding_with_cp1252)

def resolve_encodings_and_normalize(text: str) -> str:
    text = (text.encode("raw_unicode_escape").decode("utf-8", errors = "replace_decoding_with_cp1252").encode("cp1252", errors = "replace_encoding_with_utf8").decode("utf-8", errors = "replace_decoding_with_cp1252"))
    text = unidecode(text)
    return text

In [ ]:
test['discourse_text'] = test['discourse_text'].apply(lambda x : resolve_encodings_and_normalize(x))
test['essay_text'] = test['essay_text'].apply(lambda x : resolve_encodings_and_normalize(x))
test['text'] = test['discourse_type'] + ' ' + test['discourse_text'] + '[SEP]' + test['essay_text']
test['label'] = np.nan

#### Dataset + Dynamic padding

In [ ]:
class TestDataset(Dataset):
    def __init__(self, cfg, df):
        self.cfg = cfg
        self.text = df['text'].values
    def __len__(self): return len(self.text)
    def __getitem__(self, item):
        inputs = self.cfg.tokenizer.encode_plus(self.text[item], truncation = True, add_special_tokens = True, max_length = self.cfg.max_len)
        samples = {'input_ids': inputs['input_ids'], 'attention_mask': inputs['attention_mask'], }
        if 'token_type_ids' in inputs: samples['token_type_ids'] = inputs['token_type_ids']
        return samples

class Collate:
    def __init__(self, tokenizer, isTrain=True):
        self.isTrain = isTrain
        self.tokenizer = tokenizer

    def __call__(self, batch):
        output = dict()
        output["input_ids"] = [sample["input_ids"] for sample in batch]
        output["attention_mask"] = [sample["attention_mask"] for sample in batch]
        if self.isTrain: output["target"] = [sample["target"] for sample in batch]
        batch_max = max([len(ids) for ids in output["input_ids"]])
        if self.tokenizer.padding_side == "right":
            output["input_ids"] = [s + (batch_max - len(s)) * [self.tokenizer.pad_token_id] for s in output["input_ids"]]
            output["attention_mask"] = [s + (batch_max - len(s)) * [0] for s in output["attention_mask"]]
        else:
            output["input_ids"] = [(batch_max - len(s)) * [self.tokenizer.pad_token_id] + s for s in output["input_ids"]]
            output["attention_mask"] = [(batch_max - len(s)) * [0] + s for s in output["attention_mask"]]
        output["input_ids"] = torch.tensor(output["input_ids"], dtype=torch.long)
        output["attention_mask"] = torch.tensor(output["attention_mask"], dtype=torch.long)
        if self.isTrain: output["target"] = torch.tensor(output["target"], dtype=torch.long)
        return output

### The Model

In [ ]:
class MeanPooling(nn.Module):
    def __init__(self):
        super(MeanPooling, self).__init__()
        
    def forward(self, last_hidden_state, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9) #
        mean_embeddings = sum_embeddings / sum_mask
        return mean_embeddings

def inference_fn(test_loader, model, device):
    preds = []
    model.eval()
    model.to(device)
    tk0 = tqdm(test_loader, total=len(test_loader))
    for data in tk0:
        ids = data['input_ids'].to(device, dtype = torch.long)
        mask = data['attention_mask'].to(device, dtype = torch.long)
        with torch.no_grad():
            y_preds = model(ids, mask)
        y_preds = softmax(y_preds.to('cpu').numpy())
        preds.append(y_preds)
    predictions = np.concatenate(preds)
    return predictions

In [ ]:
class FeedBackModel(nn.Module):
    def __init__(self, model_name):
        super(FeedBackModel, self).__init__()
        self.model = AutoModel.from_pretrained(model_name)
        if CFG.gradient_checkpointing: (self.model).gradient_checkpointing_enable()
        if CFG.freezing:
            freeze((self.model).embeddings)
            freeze((self.model).encoder.layer[:2])
            CFG.after_freezed_parameters = filter(lambda parameter: parameter.requires_grad, (self.model).parameters())
        self.config = AutoConfig.from_pretrained(model_name)
        self.drop = nn.Dropout(p=CFG.dropout)
        self.pooler = MeanPooling()
        self.fc = nn.Linear(self.config.hidden_size, CFG.target_size)
        
    def forward(self, ids, mask):
        out = self.model(input_ids = ids, attention_mask = mask, output_hidden_states = False)
        out = self.pooler(out.last_hidden_state, mask)
        out = self.drop(out)
        outputs = self.fc(out)
        return outputs

### Deberta-Base Inference

In [ ]:
testDataset = TestDataset(CFG, test)
test_loader = DataLoader(
                          testDataset,
                          shuffle = False,
                          drop_last = False,
                          pin_memory = True,
                          batch_size = CFG.batch_size,
                          num_workers = CFG.num_workers,
                          collate_fn = Collate(CFG.tokenizer, isTrain = False)
                        )

deberta_predictions = []
for i in CFG.trn_fold:
    model = FeedBackModel(CFG.model)
    model.load_state_dict(torch.load('../input/dbv3basemodels202279/models-deberta-v3-base-deberta-v3-base_fold' + str(i) +'_best.pth'))
    prediction = inference_fn(test_loader, model, device)
    deberta_predictions.append(prediction)
    torch.cuda.empty_cache()
    gc.collect()

deb_adequate = []
deb_effective = []
deb_ineffective = []

for x in deberta_predictions:
    deb_ineffective.append(x[:, 0])
    deb_adequate.append(x[:, 1])
    deb_effective.append(x[:, 2])

deb_ineffective = pd.DataFrame(deb_ineffective).T
deb_adequate = pd.DataFrame(deb_adequate).T
deb_effective = pd.DataFrame(deb_effective).T

## Model 2: Deberta-Large

#### Configurations

In [ ]:
class CFG:
    seed = 42
    n_fold = 4
    max_len = 512
    batch_size = 32
    num_workers = 2
    model = "microsoft/deberta-large"
    path = "../input/feedback-deberta-large-051/"
    config_path = "../input/feedback-deberta-large-051/" + 'config.pth'

#### Helper Functions

In [ ]:
def replace_encoding_with_utf8(error: UnicodeError) -> Tuple[bytes, int]: return error.object[error.start : error.end].encode("utf-8"), error.end
def replace_decoding_with_cp1252(error: UnicodeError) -> Tuple[str, int]: return error.object[error.start : error.end].decode("cp1252"), error.end
codecs.register_error("replace_encoding_with_utf8", replace_encoding_with_utf8)
codecs.register_error("replace_decoding_with_cp1252", replace_decoding_with_cp1252)

def resolve_encodings_and_normalize(text: str) -> str:
    text = (text.encode("raw_unicode_escape").decode("utf-8", errors = "replace_decoding_with_cp1252").encode("cp1252", errors = "replace_encoding_with_utf8").decode("utf-8", errors = "replace_decoding_with_cp1252"))
    text = unidecode(text)
    return text

def fetch_essay(essay_id: str, txt_dir: str):
    essay_path = os.path.join(COMP_DIR + txt_dir, essay_id + '.txt')
    essay_text = open(essay_path, 'r').read()
    return essay_text

def prepare_input(cfg, text, text_2=None):
    inputs = cfg.tokenizer(text, text_2, padding = "max_length", add_special_tokens = True, max_length = cfg.max_len, truncation = True)
    for k, v in inputs.items(): inputs[k] = torch.tensor(v, dtype=torch.long)
    return inputs

def inference_fn(test_loader, model, device):
    preds = []
    model.eval()
    model.to(device)
    tk0 = tqdm(test_loader, total=len(test_loader))
    for inputs in tk0:
        for k, v in inputs.items():
            inputs[k] = v.to(device)
        with torch.no_grad():
            output = model(inputs)
        preds.append(F.softmax(output).to('cpu').numpy())
    return np.concatenate(preds)

def show_gradient(df, n_row=None):
    if not n_row: n_row = 5
    return df.head(n_row).assign(all_mean=lambda x: x.mean(axis=1)).style.background_gradient(cmap=cm, axis=1)

#### Data Loading

In [ ]:
N_ROW = 10

pd.set_option('display.precision', 4)
cm = sns.light_palette('green', as_cmap=True)
props_param = "color:white; font-weight:bold; background-color:green;"
COMP_DIR = "../input/feedback-prize-effectiveness/"
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test_path = COMP_DIR + "test.csv"
submission_path = COMP_DIR + "sample_submission.csv"
test_origin = pd.read_csv(test_path)
submission_origin = pd.read_csv(submission_path)
data_path = "../input/feedback-prize-effectiveness/train.csv"
cols_list = ['essay_id', 'discourse_text']
idxs_list = [49, 80, 945, 947, 1870]
temp = pd.read_csv(data_path, usecols=cols_list).loc[idxs_list, :]

In [ ]:
temp['discourse_text_UPD'] = temp['discourse_text'].apply(resolve_encodings_and_normalize)
temp['essay_text'] = temp['essay_id'].transform(fetch_essay, txt_dir='train')
temp['essay_text_UPD'] = temp['essay_text'].apply(resolve_encodings_and_normalize)

In [ ]:
for n, row in enumerate(temp.iterrows()):
    indx, data = row
    disc_text = data.discourse_text
    disc_text_upd = data.discourse_text_UPD
    print(f'\nN{n} === index: {indx} ===')
    print(f'\n>>> origin text:')
    print(repr(disc_text))
    print(f'\n>>> updated text:')
    print(repr(disc_text_upd))

#### Deberta Large - The Model

In [ ]:
class TestDataset(Dataset):
    def __init__(self, cfg, df):
        self.cfg = cfg
        self.text = df['text'].values
    def __len__(self): return len(self.text)
    def __getitem__(self, item):
        text = self.text[item]
        inputs = prepare_input(self.cfg, text)
        return inputs

class CustomModel(nn.Module):
    def __init__(self, cfg, config_path=None, pretrained=False):
        super().__init__()
        self.cfg = cfg
        if config_path is None: self.config = AutoConfig.from_pretrained(cfg.model, output_hidden_states=True)
        else: self.config = torch.load(config_path)
        if pretrained: self.model = AutoModel.from_pretrained(cfg.model, config=self.config)
        else: self.model = AutoModel.from_config(self.config)
        self.bilstm = nn.LSTM(self.config.hidden_size, (self.config.hidden_size) // 2, num_layers=2, dropout=self.config.hidden_dropout_prob, batch_first=True, bidirectional=True)
        self.dropout1 = nn.Dropout(0.1)
        self.dropout2 = nn.Dropout(0.2)
        self.dropout3 = nn.Dropout(0.3)
        self.dropout4 = nn.Dropout(0.4)
        self.dropout5 = nn.Dropout(0.5)
        self.output = nn.Sequential( nn.Linear(self.config.hidden_size, 3) )
                
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)

    def forward(self, inputs):
        sequence_output = self.model(**inputs)[0][:, 0, :]
        logits1 = self.output(self.dropout1(sequence_output))
        logits2 = self.output(self.dropout2(sequence_output))
        logits3 = self.output(self.dropout3(sequence_output))
        logits4 = self.output(self.dropout4(sequence_output))
        logits5 = self.output(self.dropout5(sequence_output))
        logits = (logits1 + logits2 + logits3 + logits4 + logits5) / 5
        return logits

In [ ]:
CFG.tokenizer = AutoTokenizer.from_pretrained(CFG.path + 'tokenizer')

df = test_origin.copy()
SEP = CFG.tokenizer.sep_token
df['discourse_text'] = df['discourse_text'].apply(resolve_encodings_and_normalize)
df['essay_text'] = df['essay_id'].transform(fetch_essay, txt_dir='test')
df['essay_text'] = df['essay_text'].apply(resolve_encodings_and_normalize)
df['text'] = df['discourse_type'] + ' ' + df['discourse_text'] + SEP + df['essay_text']

In [ ]:
test_dataset = TestDataset(CFG, df)
test_loader = DataLoader(test_dataset, batch_size = CFG.batch_size, shuffle = False, num_workers = CFG.num_workers, pin_memory = True, drop_last = False)

### Deberta-Large Inference

In [ ]:
deberta_large_predictions = []
for fold in range(CFG.n_fold):
    model = CustomModel(CFG, config_path=CFG.config_path, pretrained=False)
    state = torch.load(CFG.path + f"{CFG.model.replace('/', '-')}_fold{fold}_best.pth", map_location=torch.device('cpu'))
    model.load_state_dict(state['model'])
    prediction = inference_fn(test_loader, model, DEVICE)
    deberta_large_predictions.append(prediction)
    del model, state, prediction; gc.collect()
    torch.cuda.empty_cache()

deb_large_adequate = []
deb_large_effective = []
deb_large_ineffective = []

for x in deberta_large_predictions:
    deb_large_ineffective.append(x[:, 0])
    deb_large_adequate.append(x[:, 1])
    deb_large_effective.append(x[:, 2])

deb_large_ineffective = pd.DataFrame(deb_large_ineffective).T
deb_large_adequate = pd.DataFrame(deb_large_adequate).T
deb_large_effective = pd.DataFrame(deb_large_effective).T

# ReBERTa Models
_____

Couple of years ago when the a paper was published with the claim: **The original BERT architecture can outperform any other model released after it**.

Shocking. right? 

The authors managed to do it using some heavy hyperparameters search and many other training scheme tricks. They simply named their method **R**obustly **O**ptimized **BERT** pretraining **A**pproach or simply **RoBERTa**.
Which to this day remain as one of the top high performing transformers available.


**In short: RoBERTa's Hyperparameters change from BERT**

- Longer training time.
- Larger training data (x10, from 16G to 160GB).
- Larger batch size (from 256 to 8k).
- The removal of the NSP task.
- Bigger vocabulary size (from 30k to 50k).
- Longer sequences are used as input (but still keep the limitation of 512 tokens).
- Dynamic masking.

## Model 3: Roberta-Large

#### Configurations

In [ ]:
class CFG:
    n_fold = 5
    batch = 16
    max_len = 512
    num_workers = 2
    path = "../input/robertalarge"

In [ ]:
class TestDataset(Dataset):
    def __init__(self, cfg, df):
        self.cfg = cfg
        self.essay = df['essay'].values
        self.discourse = df['discourse'].values

    def __len__(self): return len(self.discourse)
    
    def __getitem__(self, item):
        discourse = self.discourse[item]
        essay = self.essay[item]
        inputs = prepare_input(self.cfg, discourse, essay)
        return inputs
        
class FeedBackModel(nn.Module):
    def __init__(self, model_path):
        super(FeedBackModel, self).__init__()
        self.model = AutoModel.from_pretrained(model_path)
        self.linear = nn.Linear(1024, 3)

    def forward(self, inputs):
        last_hidden_states = self.model(**inputs)[0][:, 0, :]
        outputs = self.linear(last_hidden_states)
        return outputs

In [ ]:
CFG.tokenizer = AutoTokenizer.from_pretrained(CFG.path)

df = test_origin.copy()

txt_sep = " "
df['discourse'] = df['discourse_type'].str.strip() + txt_sep + df['discourse_text'].str.strip()
df['essay'] = df['essay_id'].transform(fetch_essay, txt_dir='test').str.strip()

test_dataset = TestDataset(CFG, df)
test_loader = DataLoader(test_dataset, batch_size=CFG.batch, shuffle=False, num_workers=CFG.num_workers, pin_memory=True, drop_last=False)

### Roberta-Large Inference

In [ ]:
gc.collect()
roberta_predicts = []
for model_path in os.listdir('../input/feedback-roberta-models'):
    if 'data_1' in model_path:
        model = pkl.load(open('../input/feedback-roberta-models/' + model_path, 'rb'))    
        prediction = inference_fn(test_loader, model, DEVICE)
        roberta_predicts.append(prediction)
        del model, prediction
        torch.cuda.empty_cache()    
        gc.collect()
gc.collect()

rob_adequate = []
rob_effective = []
rob_ineffective = []

for x in roberta_predicts:
    rob_ineffective.append(x[:, 0])
    rob_adequate.append(x[:, 1])
    rob_effective.append(x[:, 2])

rob_ineffective = pd.DataFrame(rob_ineffective).T
rob_adequate = pd.DataFrame(rob_adequate).T
rob_effective = pd.DataFrame(rob_effective).T

# Model 4: LightGBM

#### General Configurations

In [ ]:
class CFG:
    seed = 42
    n_folds = 4

#### LightGBM Hyperparameters

In [ ]:
num_rounds = 1000    

params = {}
params['num_class'] = 3
params["max_depth"] = 10
params["verbosity"] = -1
params["num_leaves"] = 52
params['boosting'] = 'gbdt'
params["bagging_freq"] = 8
params["random_state"] = 42
params["bagging_seed"] = 10
params["lambda_l2"] = 0.0256
params['is_unbalance'] = True
params["learning_rate"] = 0.05
params["min_data_in_leaf"] = 10
params['metric'] = 'multi_logloss'
params["objective"] = 'multiclass'
params["feature_fraction"] = 0.503
params["bagging_fraction"] = 0.741

#### Data Loading

In [ ]:
INPUT_DIR = "../input/feedback-prize-effectiveness/"

def get_train_essay(essay_id):
    essay_path = os.path.join(INPUT_DIR,f'train/{essay_id}.txt')
    essay_text = open(essay_path,'r').read()
    return essay_text

def get_test_essay(essay_id):
    essay_path = os.path.join(INPUT_DIR,f'test/{essay_id}.txt')
    essay_text = open(essay_path,'r').read()
    return essay_text

train = pd.read_csv(INPUT_DIR+'train.csv')
test = pd.read_csv(INPUT_DIR+'test.csv')
train['essay_text'] = train['essay_id'].apply(get_train_essay)
test['essay_text'] = test['essay_id'].apply(get_test_essay)

def set_seed(seed=42):
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(CFG.seed)

effectiveness_map = {'Ineffective':0, 'Adequate':1, 'Effective':2}
train['target'] = train['discourse_effectiveness'].map(effectiveness_map)

for fold, (_,val_idx) in enumerate(StratifiedGroupKFold(n_splits=CFG.n_folds,shuffle=True,random_state=CFG.seed).split(X=train, y=train['target'], groups=train.essay_id)):
    train.loc[val_idx,'kfold'] = fold
word2vec_model = gensim.models.KeyedVectors.load_word2vec_format('../input/google-news/GoogleNews-vectors-negative300.bin', binary=True)

def avg_feature_vector(sentence, model, num_features):
    words = sentence.replace('\n'," ").replace(',',' ').replace('.'," ").split()
    feature_vec = np.zeros((num_features,),dtype="float32")
    i=0
    for word in words:
        try: feature_vec = np.add(feature_vec, model[word])
        except KeyError as error:
            feature_vec
            i = i + 1
    if len(words) > 0:
        feature_vec = np.divide(feature_vec, len(words)- i)
    return feature_vec

#### LightGBM Training & Inference

In [ ]:
oof_score = 0
y_test_pred = np.zeros((test.shape[0], 3))

for fold in range(CFG.n_folds):
    print(f'=============fold:{fold}==================')
    train_fold = train[train['kfold']!=fold].reset_index(drop=True)
    valid_fold = train[train['kfold']==fold].reset_index(drop=True)

    word2vec_train_disc_text = np.zeros((len(train_fold.index),300),dtype="float32")
    word2vec_valid_disc_text = np.zeros((len(valid_fold.index),300),dtype="float32")
    word2vec_test_disc_text = np.zeros((len(test.index),300),dtype="float32")
    for i in range(len(train_fold.index)): word2vec_train_disc_text[i] = avg_feature_vector(train_fold["discourse_text"][i], word2vec_model, 300)
    for i in range(len(valid_fold.index)): word2vec_valid_disc_text[i] = avg_feature_vector(valid_fold["discourse_text"][i], word2vec_model, 300)
    for i in range(len(test.index)): word2vec_test_disc_text[i] = avg_feature_vector(test["discourse_text"][i], word2vec_model, 300)

    word2vec_train_essay_text = np.zeros((len(train_fold.index),300),dtype="float32")
    word2vec_valid_essay_text = np.zeros((len(valid_fold.index),300),dtype="float32")
    word2vec_test_essay_text = np.zeros((len(test.index),300),dtype="float32")
    for i in range(len(train_fold.index)): word2vec_train_essay_text[i] = avg_feature_vector(train_fold["essay_text"][i], word2vec_model, 300)
    for i in range(len(valid_fold.index)): word2vec_valid_essay_text[i] = avg_feature_vector(valid_fold["essay_text"][i], word2vec_model, 300)
    for i in range(len(test.index)): word2vec_test_essay_text[i] = avg_feature_vector(test["essay_text"][i], word2vec_model, 300)

    ohe = OneHotEncoder()
    train_type_ohe = sparse.csr_matrix(ohe.fit_transform(train_fold['discourse_type'].values.reshape(-1,1)))
    valid_type_ohe = sparse.csr_matrix(ohe.transform(valid_fold['discourse_type'].values.reshape(-1,1)))
    test_type_ohe = sparse.csr_matrix(ohe.transform(test['discourse_type'].values.reshape(-1,1)))

    Xtrain_word2vec = sparse.hstack((train_type_ohe,word2vec_train_disc_text,word2vec_train_essay_text))
    Xvalid_word2vec = sparse.hstack((valid_type_ohe,word2vec_valid_disc_text,word2vec_valid_essay_text))
    test_word2vec = sparse.hstack((test_type_ohe,word2vec_test_disc_text,word2vec_test_essay_text))

    #lgbm
    lgtrain = lgb.Dataset(Xtrain_word2vec, label=train_fold['target'].ravel())
    lgvalidation = lgb.Dataset(Xvalid_word2vec, label=valid_fold['target'].ravel())

    model = lgb.train(params, lgtrain, num_rounds, valid_sets=[lgtrain, lgvalidation], early_stopping_rounds=100, verbose_eval=100)
    y_pred = model.predict(Xvalid_word2vec, num_iteration=model.best_iteration)
    y_test_pred += model.predict(test_word2vec, num_iteration=model.best_iteration)

    score = log_loss(valid_fold['target'], y_pred)
    oof_score += score

    print(f'Fold:{fold},valid score:{score}')

#### LightGBM Predictions Aggregation

In [ ]:
y_test_pred = y_test_pred / float(CFG.n_folds)
oof_score /= float(CFG.n_folds)
print("Aggregate OOF Score: {}".format(oof_score))

lgbm_adequate = y_test_pred[:,1]
lgbm_effective = y_test_pred[:,2]
lgbm_ineffective = y_test_pred[:,0]

lgbm_adequate = pd.DataFrame(lgbm_adequate)
lgbm_effective = pd.DataFrame(lgbm_effective)
lgbm_ineffective = pd.DataFrame(lgbm_ineffective)

# Ensembling

In [ ]:
submission = pd.read_csv('../input/feedback-prize-effectiveness/sample_submission.csv')

ineffective_ = pd.concat([deb_ineffective, deb_large_ineffective, rob_ineffective, lgbm_ineffective], keys = MODEL_NAMES, axis = 1)
adequate_ = pd.concat([deb_adequate, deb_large_adequate, rob_adequate, lgbm_adequate], keys = MODEL_NAMES, axis = 1)
effective_ = pd.concat([deb_effective, deb_large_effective, rob_effective, lgbm_effective], keys = MODEL_NAMES, axis = 1)

In [ ]:
show_gradient(ineffective_, N_ROW)
show_gradient(adequate_, N_ROW)
show_gradient(effective_, N_ROW)

In [ ]:
w_ = WEIGHTS
d_ = [('Ineffective', ineffective_), ('Adequate', adequate_), ('Effective', effective_)]

for x in d_:
    col_name, df = x
    submission[col_name] = pd.DataFrame( {col: df[col].mean(axis=1) for col in MODEL_NAMES} ).mul(w_).sum(axis=1)

submission.head(N_ROW)
submission.to_csv('submission.csv',index=False)